In [1]:
import os
import sys

# Install dependencies
!pip install --upgrade bitsandbytes
!pip install --upgrade transformers trl

# Explicitly force bitsandbytes to look for the 128 binary
os.environ["BNB_CUDA_VERSION"] = "128"

# Ensure Colab's CUDA paths are visible to the library loader
os.environ["LD_LIBRARY_PATH"] = "/usr/local/cuda-12.8/lib64:" + os.environ.get("LD_LIBRARY_PATH", "")

# Now import to initialize
import bitsandbytes as bnb
import torch

print(f"CUDA Available: {torch.cuda.is_available()}")
print(f"GPU Device: {torch.cuda.get_device_name(0)}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 40.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 107.6 MB/s eta 0:00:0000:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 751.0/751.0 kB 57.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 52.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 50.3 MB/s eta 0:00:00:00:0100:01
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


This can be used to load a bitsandbytes version built with a CUDA version that is different from the PyTorch CUDA version.
If this was unintended set the BNB_CUDA_VERSION variable to an empty string: export BNB_CUDA_VERSION=



CUDA Available: True
GPU Device: NVIDIA A100-SXM4-40GB


In [2]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

TRAIN_PATH = '/content/drive/MyDrive/trino-data/train.jsonl'
VAL_PATH = '/content/drive/MyDrive/trino-data/val.jsonl'
TEST_PATH = '/content/drive/MyDrive/trino-data/test.jsonl'

print("Paths set.")

Mounted at /content/drive
Paths set.


In [3]:
# Load and verify data
from datasets import load_dataset

train_dataset = load_dataset('json', data_files=TRAIN_PATH, split='train')
val_dataset = load_dataset('json', data_files=VAL_PATH, split='train')

print(f"Train: {len(train_dataset)} rows")
print(f"Val: {len(val_dataset)} rows")
print("\nSample row:")
print(train_dataset[0])

Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

Train: 15177 rows
Val: 1837 rows

Sample row:
{'messages': [{'role': 'system', 'content': 'You are a senior engineer reviewing code at Trino.'}, {'role': 'user', 'content': 'Review this diff:\n\nplugin/trino-iceberg/src/main/java/io/trino/plugin/iceberg/procedure/RollbackToSnapshotProcedure.java: @@ -31,6 +31,7 @@\n import static java.lang.invoke.MethodHandles.lookup;\n import static java.util.Objects.requireNonNull;\n \n+@Deprecated'}, {'role': 'assistant', 'content': 'Is this part of a larger endeavor ?\r\nIf there are other follow-ups maybe it is worth creating a epic issue to track what else will be done.'}]}


In [4]:
# Load model and apply QLoRA
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

MODEL_ID = "codellama/CodeLlama-13b-instruct-hf"

# Your existing bnb config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.add_special_tokens({'pad_token': '[PAD]'})
tokenizer.padding_side = "right"

print("Loading model in 4-bit...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    
    # FIX 1: Explicitly force the entire model onto GPU 0 
    # instead of letting accelerate guess with "auto"
    device_map={"": 0}, 
    
    # FIX 2: Prevent loading weights into full float32 first
    torch_dtype=torch.bfloat16, 
    
    # FIX 3: Low CPU memory usage initialization
    low_cpu_mem_usage=True 
)


# Resize token embeddings in case a new pad token was added
model.resize_token_embeddings(len(tokenizer))

# Configure model for training
model.config.use_cache = False  # Silence warnings, must be False for training
model = prepare_model_for_kbit_training(model)

# Expanded target modules for optimal QLoRA convergence
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=[
        "q_proj", 
        "k_proj", 
        "v_proj", 
        "o_proj", 
        "gate_proj", 
        "up_proj", 
        "down_proj"
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

Loading tokenizer...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/589 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

...

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/363 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

[transformers] The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`
[transformers] The new lm_head weights will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


trainable params: 62,586,880 || all params: 13,078,635,520 || trainable%: 0.4785


In [8]:
# Train
from trl import SFTTrainer
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="/content/drive/MyDrive/candor-checkpoints",
    num_train_epochs=2,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    warmup_steps=50,
    learning_rate=2e-4,
    bf16=True,
    optim="paged_adamw_8bit",
    logging_steps=25,
    eval_strategy="steps",
    eval_steps=200,
    save_strategy="steps",
    save_steps=200,
    load_best_model_at_end=True,
    report_to="none"
)

def formatting_func(example):
    text = ""
    for msg in example["messages"]:
        if msg["role"] == "system":
            text += f"<s>[INST] <<SYS>>\n{msg['content']}\n<</SYS>>\n\n"
        elif msg["role"] == "user":
            text += f"{msg['content']} [/INST] "
        elif msg["role"] == "assistant":
            text += f"{msg['content']} </s>"
    return text

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
    formatting_func=formatting_func,
)

trainer.train()

In [7]:
import shutil

shutil.copytree(
    "/content/drive/MyDrive/candor-checkpoints/checkpoint-1898",
    "/content/drive/MyDrive/candor-checkpoints/final-adapter"
)
print("Final adapter saved.")

Final adapter saved.
